# 001_Prompt_Evals_complete.ipynb — Amazon Bedrock

Genera un dataset y evalúa soluciones AWS con notas de sintaxis y del modelo. Las instrucciones JSON sustituyen los prefills.

Ejecuta las celdas en orden con credenciales AWS y acceso al modelo. La configuración usa servicios reales. [Configuración](../README.md).

In [ ]:
# Cliente real de Amazon Bedrock: credenciales AWS del entorno o perfil.
import os
from pathlib import Path

import boto3
from dotenv import load_dotenv

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "BEDROCK" / "README.md").is_file()
)
bedrock_root = repo_root / "BEDROCK"
load_dotenv(bedrock_root / ".env", override=False)
exercise_dir = bedrock_root / "PromptEvalsBuildingWithApi"
os.chdir(exercise_dir)

session = boto3.Session(
    profile_name=os.getenv("AWS_PROFILE"),
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)
client = session.client("bedrock-runtime")
model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-6")

import json


In [ ]:
# Mensajes y respuestas en el formato nativo de Converse.
def message_content(content):
    if isinstance(content, str):
        if not content.strip():
            raise ValueError("El texto del mensaje no puede estar vacío.")
        return [{"text": content}]
    if not isinstance(content, list) or not content:
        raise ValueError("El mensaje necesita una lista de bloques de contenido.")
    for block in content:
        if "text" in block and not block["text"].strip():
            raise ValueError("Los bloques de texto no pueden estar vacíos.")
    return content


def add_user_message(messages, content):
    messages.append({"role": "user", "content": message_content(content)})


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": message_content(content)})


def chat(
    messages,
    system=None,
    stop_sequences=None,
    tools=None,
    tool_choice="auto",
    max_tokens=4000,
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {"maxTokens": max_tokens},
    }
    if stop_sequences:
        params["inferenceConfig"]["stopSequences"] = list(stop_sequences)
    if system:
        params["system"] = [{"text": system}]
    if tools:
        choices = {"auto": {"auto": {}}, "any": {"any": {}}}
        choice = choices.get(tool_choice, {"tool": {"name": tool_choice}})
        selected_tools = list(tools)
        params["toolConfig"] = {"tools": selected_tools, "toolChoice": choice}
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]
    text = "\n".join(part["text"] for part in parts if "text" in part)
    return text


In [ ]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system="Return only a valid JSON array. Do not include Markdown fences or commentary.")
    return json.loads(text)


In [ ]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with {test_case["format"]}; do not use Markdown fences
* Do not add any comments or commentary or explanation
"""

    system_prompt = "You are an experienced AWS engineer with a hyper focus on addressing corner cases"

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages, system=system_prompt)
    return output


In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": ["A key strength"],
    "weaknesses": ["An area for improvement"],
    "reasoning": "Brief assessment",
    "score": 8
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, system="Return only a valid JSON object. Do not include Markdown fences or commentary.")
    return json.loads(eval_text)


In [ ]:
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [ ]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))